# Replay verification, no model

Applies 8 patches (that resolved LOCALLY) to the base snapshots, applies the official
test_patch, and runs exactly the test nodes swegemma would run. CPU only.

Purpose: is the local sandbox the reason these tasks resolve locally but not on Kaggle?


In [ ]:

import base64, json, os, subprocess, sys
from pathlib import Path

print('input:', sorted(os.listdir('/kaggle/input')))
TASKS = None
for root, dirs, files in os.walk('/kaggle/input'):
    if 'tasks.jsonl' in files:
        TASKS = Path(root) / 'tasks.jsonl'
        break
SNAPS = None
for root, dirs, files in os.walk('/kaggle/input'):
    if any(f.endswith('.tgz') for f in files):
        SNAPS = Path(root)
        break
print('tasks:', TASKS)
print('snaps:', SNAPS)
assert TASKS and SNAPS

PATCHES = json.loads("""{"fastapi_14301": "ZGlmZiAtLWdpdCBhL2Zhc3RhcGkvZGVwZW5kZW5jaWVzL3V0aWxzLnB5IGIvZmFzdGFwaS9kZXBlbmRlbmNpZXMvdXRpbHMucHkKaW5kZXggYzVjNmI2OWJiLi5mNzFlODRlZTMgMTAwNjQ0Ci0tLSBhL2Zhc3RhcGkvZGVwZW5kZW5jaWVzL3V0aWxzLnB5CisrKyBiL2Zhc3RhcGkvZGVwZW5kZW5jaWVzL3V0aWxzLnB5CkBAIC0xMjksNyArMTI5LDcgQEAgZGVmIGdldF9wYXJhbWV0ZXJsZXNzX3N1Yl9kZXBlbmRhbnQoKiwgZGVwZW5kczogcGFyYW1zLkRlcGVuZHMsIHBhdGg6IHN0cikgLT4gRGUKICAgICBpZiBpc2luc3RhbmNlKGRlcGVuZHMsIHBhcmFtcy5TZWN1cml0eSkgYW5kIGRlcGVuZHMuc2NvcGVzOgogICAgICAgICB1c2Vfc2VjdXJpdHlfc2NvcGVzLmV4dGVuZChkZXBlbmRzLnNjb3BlcykKICAgICByZXR1cm4gZ2V0X2RlcGVuZGFudCgKLSAgICAgICAgcGF0aD1wYXRoLCBjYWxsPWRlcGVuZHMuZGVwZW5kZW5jeSwgc2VjdXJpdHlfc2NvcGVzPXVzZV9zZWN1cml0eV9zY29wZXMKKyAgICAgICAgcGF0aD1wYXRoLCBjYWxsPWRlcGVuZHMuZGVwZW5kZW5jeSwgc2VjdXJpdHlfc2NvcGVzPXVzZV9zZWN1cml0eV9zY29wZXMsIHNjb3BlPWRlcGVuZHMuc2NvcGUKICAgICApCiAKIAo=", "fastapi_14463": "ZGlmZiAtLWdpdCBhL2Zhc3RhcGkvb3BlbmFwaS91dGlscy5weSBiL2Zhc3RhcGkvb3BlbmFwaS91dGlscy5weQppbmRleCA3NWZmMjYxMDIuLjc5YmYwZTExMiAxMDA2NDQKLS0tIGEvZmFzdGFwaS9vcGVuYXBpL3V0aWxzLnB5CisrKyBiL2Zhc3RhcGkvb3BlbmFwaS91dGlscy5weQpAQCAtMSw2ICsxLDcgQEAKIGltcG9ydCBodHRwLmNsaWVudAogaW1wb3J0IGluc3BlY3QKIGltcG9ydCB3YXJuaW5ncworaW1wb3J0IGNvcHkKIGZyb20gY29sbGVjdGlvbnMuYWJjIGltcG9ydCBTZXF1ZW5jZQogZnJvbSB0eXBpbmcgaW1wb3J0IEFueSwgT3B0aW9uYWwsIFVuaW9uLCBjYXN0CiAKQEAgLTM3Nyw3ICszNzgsNyBAQCBkZWYgZ2V0X29wZW5hcGlfcGF0aCgKICAgICAgICAgICAgICAgICAgICAgYWRkaXRpb25hbF9zdGF0dXNfY29kZSwKICAgICAgICAgICAgICAgICAgICAgYWRkaXRpb25hbF9yZXNwb25zZSwKICAgICAgICAgICAgICAgICApIGluIHJvdXRlLnJlc3BvbnNlcy5pdGVtcygpOgotICAgICAgICAgICAgICAgICAgICBwcm9jZXNzX3Jlc3BvbnNlID0gYWRkaXRpb25hbF9yZXNwb25zZS5jb3B5KCkKKyAgICAgICAgICAgICAgICAgICAgcHJvY2Vzc19yZXNwb25zZSA9IGNvcHkuZGVlcGNvcHkoYWRkaXRpb25hbF9yZXNwb25zZSkKICAgICAgICAgICAgICAgICAgICAgcHJvY2Vzc19yZXNwb25zZS5wb3AoIm1vZGVsIiwgTm9uZSkKICAgICAgICAgICAgICAgICAgICAgc3RhdHVzX2NvZGVfa2V5ID0gc3RyKGFkZGl0aW9uYWxfc3RhdHVzX2NvZGUpLnVwcGVyKCkKICAgICAgICAgICAgICAgICAgICAgaWYgc3RhdHVzX2NvZGVfa2V5ID09ICJERUZBVUxUIjoK", "fastapi_14616": "ZGlmZiAtLWdpdCBhL2Zhc3RhcGkvZGVwZW5kZW5jaWVzL3V0aWxzLnB5IGIvZmFzdGFwaS9kZXBlbmRlbmNpZXMvdXRpbHMucHkKaW5kZXggZTVhYzUxZDUzLi4wMDg2YzNmNzYgMTAwNjQ0Ci0tLSBhL2Zhc3RhcGkvZGVwZW5kZW5jaWVzL3V0aWxzLnB5CisrKyBiL2Zhc3RhcGkvZGVwZW5kZW5jaWVzL3V0aWxzLnB5CkBAIC03MzEsNyArNzMxLDcgQEAgZGVmIF9nZXRfbXVsdGlkaWN0X3ZhbHVlKAogKSAtPiBBbnk6CiAgICAgYWxpYXMgPSBhbGlhcyBvciBnZXRfdmFsaWRhdGlvbl9hbGlhcyhmaWVsZCkKICAgICBpZiBpc19zZXF1ZW5jZV9maWVsZChmaWVsZCkgYW5kIGlzaW5zdGFuY2UodmFsdWVzLCAoSW1tdXRhYmxlTXVsdGlEaWN0LCBIZWFkZXJzKSk6Ci0gICAgICAgIHZhbHVlID0gdmFsdWVzLmdldGxpc3QoYWxpYXMpCisgICAgICAgIHZhbHVlID0gdmFsdWVzLmdldChhbGlhcykKICAgICBlbHNlOgogICAgICAgICB2YWx1ZSA9IHZhbHVlcy5nZXQoYWxpYXMsIE5vbmUpCiAgICAgaWYgKApAQCAtNzk3LDcgKzc5Nyw3IEBAIGRlZiByZXF1ZXN0X3BhcmFtc190b19hcmdzKAogICAgIGZvciBrZXkgaW4gcmVjZWl2ZWRfcGFyYW1zLmtleXMoKToKICAgICAgICAgaWYga2V5IG5vdCBpbiBwcm9jZXNzZWRfa2V5czoKICAgICAgICAgICAgIGlmIGhhc2F0dHIocmVjZWl2ZWRfcGFyYW1zLCAiZ2V0bGlzdCIpOgotICAgICAgICAgICAgICAgIHZhbHVlID0gcmVjZWl2ZWRfcGFyYW1zLmdldGxpc3Qoa2V5KQorICAgICAgICAgICAgICAgIHZhbHVlID0gcmVjZWl2ZWRfcGFyYW1zLmdldChrZXkpCiAgICAgICAgICAgICAgICAgaWYgaXNpbnN0YW5jZSh2YWx1ZSwgbGlzdCkgYW5kIChsZW4odmFsdWUpID09IDEpOgogICAgICAgICAgICAgICAgICAgICBwYXJhbXNfdG9fcHJvY2Vzc1trZXldID0gdmFsdWVbMF0KICAgICAgICAgICAgICAgICBlbHNlOgpAQCAtOTE1LDcgKzkxNSw3IEBAIGFzeW5jIGRlZiBfZXh0cmFjdF9mb3JtX2JvZHkoCiAgICAgZmllbGRfYWxpYXNlcyA9IHtnZXRfdmFsaWRhdGlvbl9hbGlhcyhmaWVsZCkgZm9yIGZpZWxkIGluIGJvZHlfZmllbGRzfQogICAgIGZvciBrZXkgaW4gcmVjZWl2ZWRfYm9keS5rZXlzKCk6CiAgICAgICAgIGlmIGtleSBub3QgaW4gZmllbGRfYWxpYXNlczoKLSAgICAgICAgICAgIHBhcmFtX3ZhbHVlcyA9IHJlY2VpdmVkX2JvZHkuZ2V0bGlzdChrZXkpCisgICAgICAgICAgICBwYXJhbV92YWx1ZXMgPSByZWNlaXZlZF9ib2R5LmdldChrZXkpCiAgICAgICAgICAgICBpZiBsZW4ocGFyYW1fdmFsdWVzKSA9PSAxOgogICAgICAgICAgICAgICAgIHZhbHVlc1trZXldID0gcGFyYW1fdmFsdWVzWzBdCiAgICAgICAgICAgICBlbHNlOgo=", "fastapi_14786": "ZGlmZiAtLWdpdCBhL2Zhc3RhcGkvc2VjdXJpdHkvdXRpbHMucHkgYi9mYXN0YXBpL3NlY3VyaXR5L3V0aWxzLnB5CmluZGV4IDAwMmU2OGI0NC4uZmQzNDlhZWM3IDEwMDY0NAotLS0gYS9mYXN0YXBpL3NlY3VyaXR5L3V0aWxzLnB5CisrKyBiL2Zhc3RhcGkvc2VjdXJpdHkvdXRpbHMucHkKQEAgLTcsNCArNyw0IEBAIGRlZiBnZXRfYXV0aG9yaXphdGlvbl9zY2hlbWVfcGFyYW0oCiAgICAgaWYgbm90IGF1dGhvcml6YXRpb25faGVhZGVyX3ZhbHVlOgogICAgICAgICByZXR1cm4gIiIsICIiCiAgICAgc2NoZW1lLCBfLCBwYXJhbSA9IGF1dGhvcml6YXRpb25faGVhZGVyX3ZhbHVlLnBhcnRpdGlvbigiICIpCi0gICAgcmV0dXJuIHNjaGVtZSwgcGFyYW0KKyAgICByZXR1cm4gc2NoZW1lLCBwYXJhbS5zdHJpcCgpCg==", "fastapi_14873": "ZGlmZiAtLWdpdCBhL2Zhc3RhcGkvcm91dGluZy5weSBiL2Zhc3RhcGkvcm91dGluZy5weQppbmRleCAwYjRkMjg4NzMuLjkwZDk5OWMzNyAxMDA2NDQKLS0tIGEvZmFzdGFwaS9yb3V0aW5nLnB5CisrKyBiL2Zhc3RhcGkvcm91dGluZy5weQpAQCAtOTc5LDYgKzk3OSw4IEBAIGNsYXNzIEFQSVJvdXRlcihyb3V0aW5nLlJvdXRlcik6CiAgICAgICAgICAgICByZWRpcmVjdF9zbGFzaGVzPXJlZGlyZWN0X3NsYXNoZXMsCiAgICAgICAgICAgICBkZWZhdWx0PWRlZmF1bHQsCiAgICAgICAgICAgICBsaWZlc3Bhbj1saWZlc3Bhbl9jb250ZXh0LAorICAgICAgICAgICAgb25fc3RhcnR1cD1vbl9zdGFydHVwLAorICAgICAgICAgICAgb25fc2h1dGRvd249b25fc2h1dGRvd24sCiAgICAgICAgICkKICAgICAgICAgaWYgcHJlZml4OgogICAgICAgICAgICAgYXNzZXJ0IHByZWZpeC5zdGFydHN3aXRoKCIvIiksICJBIHBhdGggcHJlZml4IG11c3Qgc3RhcnQgd2l0aCAnLyciCg==", "fastapi_9425": "ZGlmZiAtLWdpdCBhL2Zhc3RhcGkvZGVwZW5kZW5jaWVzL3V0aWxzLnB5IGIvZmFzdGFwaS9kZXBlbmRlbmNpZXMvdXRpbHMucHkKaW5kZXggOGRkMGExNGM4Li5iZGQ5NDY0NWEgMTAwNjQ0Ci0tLSBhL2Zhc3RhcGkvZGVwZW5kZW5jaWVzL3V0aWxzLnB5CisrKyBiL2Zhc3RhcGkvZGVwZW5kZW5jaWVzL3V0aWxzLnB5CkBAIC0yNTEsNiArMjUxLDggQEAgZGVmIGdldF90eXBlZF9zaWduYXR1cmUoY2FsbDogQ2FsbGFibGVbLi4uLCBBbnldKSAtPiBpbnNwZWN0LlNpZ25hdHVyZToKIAogCiBkZWYgZ2V0X3R5cGVkX2Fubm90YXRpb24oYW5ub3RhdGlvbjogQW55LCBnbG9iYWxuczogRGljdFtzdHIsIEFueV0pIC0+IEFueToKKyAgICBpZiBhbm5vdGF0aW9uIGlzIE5vbmUgb3IgYW5ub3RhdGlvbiA9PSAiTm9uZSI6CisgICAgICAgIHJldHVybiBOb25lCiAgICAgaWYgaXNpbnN0YW5jZShhbm5vdGF0aW9uLCBzdHIpOgogICAgICAgICBhbm5vdGF0aW9uID0gRm9yd2FyZFJlZihhbm5vdGF0aW9uKQogICAgICAgICBhbm5vdGF0aW9uID0gZXZhbHVhdGVfZm9yd2FyZHJlZihhbm5vdGF0aW9uLCBnbG9iYWxucywgZ2xvYmFsbnMpCg=="}""")
PD = Path('/kaggle/working/patches'); PD.mkdir(parents=True, exist_ok=True)
for tid, b64 in PATCHES.items():
    (PD / f'{tid}.patch').write_bytes(base64.b64decode(b64))
print('patches written:', len(PATCHES))

RUNNER = Path('/kaggle/working/replay_verify2.py')
RUNNER.write_bytes(base64.b64decode("""IiIiRmFpdGhmdWwgcmVwbGF5OiBtaXJyb3JzIHN3ZWdlbW1hJ3Mgb3duIHZlcmlmaWNhdGlvbiBydWxlLgoKc3dlZ2VtbWEgZGVyaXZlcyB0aGUgcmVxdWlyZWQgdGVzdCBub2RlcyBmcm9tIHRoZSB0ZXN0X3BhdGNoCihgX2V4dHJhY3RfdGVzdF9mdW5jdGlvbnNfZnJvbV9wYXRjaGApIGFuZCBydW5zIG9ubHkgdGhvc2UsIGZhaWxpbmcgb24KZmFpbGVkL2Vycm9yL3NraXBwZWQuIFRoaXMgc2NyaXB0IGRvZXMgZXhhY3RseSB0aGUgc2FtZSwgd2l0aCBubyBtb2RlbCBpbnZvbHZlZCwKc28gdGhlIHNhbWUgcGF0Y2hlcyBjYW4gYmUgcmVwbGF5ZWQgaW4gYW55IGVudmlyb25tZW50LgoiIiIKaW1wb3J0IGpzb24KaW1wb3J0IG9zCmltcG9ydCByZQppbXBvcnQgc2h1dGlsCmltcG9ydCBzdWJwcm9jZXNzCmltcG9ydCBzeXMKaW1wb3J0IHRlbXBmaWxlCmZyb20gcGF0aGxpYiBpbXBvcnQgUGF0aAoKVEFTS1MgPSBQYXRoKHN5cy5hcmd2WzFdKQpTTkFQUyA9IFBhdGgoc3lzLmFyZ3ZbMl0pClBBVENIRElSID0gUGF0aChzeXMuYXJndlszXSkKSURTID0gc3lzLmFyZ3ZbNDpdCk9VVCA9IFBhdGgob3MuZW52aXJvbi5nZXQoJ1JFUExBWV9PVVQnLCAnL291dC9yZXBsYXlfcmVzdWx0Lmpzb24nKSkKCnRhc2tzID0ge2pzb24ubG9hZHMobClbJ2luc3RhbmNlX2lkJ106IGpzb24ubG9hZHMobCkKICAgICAgICAgZm9yIGwgaW4gVEFTS1MucmVhZF90ZXh0KGVuY29kaW5nPSd1dGYtOCcpLnNwbGl0bGluZXMoKSBpZiBsLnN0cmlwKCl9ClRFU1RGSUxFID0gcmUuY29tcGlsZShyJ15cK1wrXCsgYi8oLio/KSQnLCByZS5NKQoKCmRlZiBleHRyYWN0X3Rlc3RzKHRlc3RfcGF0Y2g6IHN0cik6CiAgICAiIiJCeXRlLWlkZW50aWNhbCBsb2dpYyB0byBzd2VnZW1tYS5oYXJuZXNzLnZlcmlmaWNhdGlvbi5fZXh0cmFjdF90ZXN0X2Z1bmN0aW9uc19mcm9tX3BhdGNoLiIiIgogICAgaWYgbm90IHRlc3RfcGF0Y2g6CiAgICAgICAgcmV0dXJuIFtdCiAgICBhZGRlZCA9IHJlLmZpbmRhbGwocideXCtccyooPzphc3luY1xzKyk/ZGVmXHMrKHRlc3RfW0EtWmEtejAtOV9dKylccypcKCcsIHRlc3RfcGF0Y2gsIHJlLk1VTFRJTElORSkKICAgIGlmIGFkZGVkOgogICAgICAgIHJldHVybiBsaXN0KGRpY3QuZnJvbWtleXMoYWRkZWQpKQogICAgcmVtb3ZlZCA9IHNldChyZS5maW5kYWxsKHInXi1ccyooPzphc3luY1xzKyk/ZGVmXHMrKHRlc3RfW0EtWmEtejAtOV9dKylccypcKCcsIHRlc3RfcGF0Y2gsIHJlLk1VTFRJTElORSkpCiAgICBodW5rX2hlYWRlcnMgPSByZS5maW5kYWxsKHInXkBAW15AXStAQFxzKig/OmFzeW5jXHMrKT9kZWZccysodGVzdF9bQS1aYS16MC05X10rKVxiJywgdGVzdF9wYXRjaCwgcmUuTVVMVElMSU5FKQogICAgY29udGV4dF9kZWZzID0gcmUuZmluZGFsbChyJ14gXHMqKD86YXN5bmNccyspP2RlZlxzKyh0ZXN0X1tBLVphLXowLTlfXSspXHMqXCgnLCB0ZXN0X3BhdGNoLCByZS5NVUxUSUxJTkUpCiAgICBjb250ZXh0X3NldCA9IHNldChjb250ZXh0X2RlZnMpCiAgICBjb21iaW5lZCA9IGxpc3QoZGljdC5mcm9ta2V5cyhodW5rX2hlYWRlcnMgKyBjb250ZXh0X2RlZnMpKQogICAgcmV0dXJuIFtmbiBmb3IgZm4gaW4gY29tYmluZWQgaWYgZm4gbm90IGluIHJlbW92ZWQgYW5kIG5vdCAoCiAgICAgICAgZm4gbm90IGluIGNvbnRleHRfc2V0IGFuZCAoYW55KHIuc3RhcnRzd2l0aChmbikgZm9yIHIgaW4gcmVtb3ZlZCkKICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgICBvciBhbnkoYyAhPSBmbiBhbmQgYy5zdGFydHN3aXRoKGZuKSBmb3IgYyBpbiBjb250ZXh0X3NldCkpKV0KCgpkZWYgc2goY21kLCBjd2QpOgogICAgcmV0dXJuIHN1YnByb2Nlc3MucnVuKGNtZCwgY3dkPXN0cihjd2QpLCBjYXB0dXJlX291dHB1dD1UcnVlLCB0ZXh0PVRydWUsCiAgICAgICAgICAgICAgICAgICAgICAgICAgZW5jb2Rpbmc9J3V0Zi04JywgZXJyb3JzPSdyZXBsYWNlJywgdGltZW91dD0xODAwKQoKCnJlc3VsdHMgPSBbXQpmb3IgdGlkIGluIElEUzoKICAgIHJlYyA9IGRpY3QodGFzaz10aWQpCiAgICBzbmFwLCBwYXRjaCA9IFNOQVBTIC8gZid7dGlkfS50Z3onLCBQQVRDSERJUiAvIGYne3RpZH0ucGF0Y2gnCiAgICB0ID0gdGFza3MuZ2V0KHRpZCwge30pCiAgICB0cF90ZXh0ID0gdC5nZXQoJ3Rlc3RfcGF0Y2gnKSBvciAnJwogICAgaWYgbm90IHNuYXAuZXhpc3RzKCk6CiAgICAgICAgcmVjWydlcnJvciddID0gJ25vIHNuYXBzaG90JwogICAgICAgIHJlc3VsdHMuYXBwZW5kKHJlYykKICAgICAgICBjb250aW51ZQogICAgd29yayA9IFBhdGgodGVtcGZpbGUubWtkdGVtcChwcmVmaXg9ZidyX3t0aWR9XycpKQogICAgdHJ5OgogICAgICAgIGlmIHNoKFsndGFyJywgJ3h6ZicsIHN0cihzbmFwKSwgJy1DJywgc3RyKHdvcmspXSwgd29yaykucmV0dXJuY29kZSAhPSAwOgogICAgICAgICAgICByZWNbJ2Vycm9yJ10gPSAnZXh0cmFjdCBmYWlsZWQnCiAgICAgICAgICAgIHJlc3VsdHMuYXBwZW5kKHJlYykKICAgICAgICAgICAgY29udGludWUKICAgICAgICBpZiBwYXRjaC5leGlzdHMoKToKICAgICAgICAgICAgcCA9IHNoKFsnZ2l0JywgJ2FwcGx5JywgJy0td2hpdGVzcGFjZT1ub3dhcm4nLCBzdHIocGF0Y2gucmVzb2x2ZSgpKV0sIHdvcmspCiAgICAgICAgICAgIHJlY1snYWdlbnRfcGF0Y2hfcmMnXSA9IHAucmV0dXJuY29kZQogICAgICAgICAgICBpZiBwLnJldHVybmNvZGUgIT0gMDoKICAgICAgICAgICAgICAgIHAgPSBzaChbJ2dpdCcsICdhcHBseScsICctLTN3YXknLCAnLS13aGl0ZXNwYWNlPW5vd2FybicsIHN0cihwYXRjaC5yZXNvbHZlKCkpXSwgd29yaykKICAgICAgICAgICAgICAgIHJlY1snYWdlbnRfcGF0Y2hfM3dheV9yYyddID0gcC5yZXR1cm5jb2RlCiAgICAgICAgICAgIHJlY1snYWdlbnRfcGF0Y2hfdGFpbCddID0gKChwLnN0ZGVyciBvciAnJykgKyAocC5zdGRvdXQgb3IgJycpKVstMjAwOl0KICAgICAgICB0cCA9IHdvcmsucGFyZW50IC8gZid7dGlkfV90ZXN0LnBhdGNoJwogICAgICAgIHRwLndyaXRlX2J5dGVzKCh0cF90ZXh0IGlmIHRwX3RleHQuZW5kc3dpdGgoJ1xuJykgZWxzZSB0cF90ZXh0ICsgJ1xuJykKICAgICAgICAgICAgICAgICAgICAgICAucmVwbGFjZSgnXHJcbicsICdcbicpLmVuY29kZSgndXRmLTgnKSkKICAgICAgICBwID0gc2goWydnaXQnLCAnYXBwbHknLCAnLS13aGl0ZXNwYWNlPW5vd2FybicsIHN0cih0cCldLCB3b3JrKQogICAgICAgIHJlY1sndGVzdF9wYXRjaF9yYyddID0gcC5yZXR1cm5jb2RlCiAgICAgICAgaWYgcC5yZXR1cm5jb2RlICE9IDA6CiAgICAgICAgICAgIHAgPSBzaChbJ2dpdCcsICdhcHBseScsICctLTN3YXknLCAnLS13aGl0ZXNwYWNlPW5vd2FybicsIHN0cih0cCldLCB3b3JrKQogICAgICAgICAgICByZWNbJ3Rlc3RfcGF0Y2hfM3dheV9yYyddID0gcC5yZXR1cm5jb2RlCiAgICAgICAgICAgIHJlY1sndGVzdF9wYXRjaF90YWlsJ10gPSAoKHAuc3RkZXJyIG9yICcnKSArIChwLnN0ZG91dCBvciAnJykpWy0yMDA6XQoKICAgICAgICBmaWxlcyA9IHNvcnRlZCh7Zi5zdHJpcCgpIGZvciBmIGluIFRFU1RGSUxFLmZpbmRhbGwodHBfdGV4dCl9KQogICAgICAgIGZucyA9IGV4dHJhY3RfdGVzdHModHBfdGV4dCkKICAgICAgICByZWNbJ3Rlc3RfZmlsZXMnXSA9IGZpbGVzCiAgICAgICAgcmVjWyd0ZXN0X2ZucyddID0gZm5zCiAgICAgICAgbm9kZXMgPSBbZid7Zn06Ontmbn0nIGZvciBmIGluIGZpbGVzIGZvciBmbiBpbiBmbnNdIG9yIGZpbGVzCiAgICAgICAgaWYgbm90IG5vZGVzOgogICAgICAgICAgICByZWNbJ2Vycm9yJ10gPSAnbm8gbm9kZXMnCiAgICAgICAgICAgIHJlc3VsdHMuYXBwZW5kKHJlYykKICAgICAgICAgICAgY29udGludWUKICAgICAgICBwID0gc2goW3N5cy5leGVjdXRhYmxlLCAnLW0nLCAncHl0ZXN0JywgJy1xJywgJy0tbm8taGVhZGVyJywgJy1wJywgJ25vOmNhY2hlcHJvdmlkZXInXSArIG5vZGVzLCB3b3JrKQogICAgICAgIHRhaWwgPSAocC5zdGRvdXQgb3IgJycpICsgKHAuc3RkZXJyIG9yICcnKQogICAgICAgIG0gPSByZS5maW5kYWxsKHInKFxkKylccysocGFzc2VkfGZhaWxlZHxlcnJvcnxlcnJvcnN8c2tpcHBlZCknLCB0YWlsKQogICAgICAgIHJlY1snY291bnRzJ10gPSB7azogaW50KHYpIGZvciB2LCBrIGluIG19CiAgICAgICAgcmVjWydub2RlcyddID0gbm9kZXMKICAgICAgICByZWNbJ3B5dGVzdF90YWlsJ10gPSB0YWlsWy0zMDA6XQogICAgICAgIHBhc3NlZCA9IHJlY1snY291bnRzJ10uZ2V0KCdwYXNzZWQnLCAwKQogICAgICAgIGJhZCA9IHN1bShyZWNbJ2NvdW50cyddLmdldChrLCAwKSBmb3IgayBpbiAoJ2ZhaWxlZCcsICdlcnJvcicsICdlcnJvcnMnLCAnc2tpcHBlZCcpKQogICAgICAgIHJlY1sncmVzb2x2ZWRfaGVyZSddID0gYm9vbChwYXNzZWQgPiAwIGFuZCBiYWQgPT0gMCkKICAgIGV4Y2VwdCBFeGNlcHRpb24gYXMgZToKICAgICAgICByZWNbJ2Vycm9yJ10gPSBmJ3t0eXBlKGUpLl9fbmFtZV9ffToge2V9JwogICAgZmluYWxseToKICAgICAgICBzaHV0aWwucm10cmVlKHdvcmssIGlnbm9yZV9lcnJvcnM9VHJ1ZSkKICAgIHJlc3VsdHMuYXBwZW5kKHJlYykKCk9VVC5wYXJlbnQubWtkaXIocGFyZW50cz1UcnVlLCBleGlzdF9vaz1UcnVlKQpPVVQud3JpdGVfdGV4dChqc29uLmR1bXBzKHJlc3VsdHMsIGluZGVudD0xKSwgZW5jb2Rpbmc9J3V0Zi04JykKcHJpbnQoJyUtMjJzICUtN3MgJS03cyAlLTlzICVzJyAlICgndGFzaycsICdhX3JjJywgJ3RfcmMnLCAncmVzb2x2ZWQnLCAnY291bnRzIC8gZXJyJykpCmZvciByIGluIHJlc3VsdHM6CiAgICBwcmludCgnJS0yMnMgJS03cyAlLTdzICUtOXMgJXMnICUgKHIuZ2V0KCd0YXNrJyksIHIuZ2V0KCdhZ2VudF9wYXRjaF9yYycpLCByLmdldCgndGVzdF9wYXRjaF9yYycpLAogICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgICByLmdldCgncmVzb2x2ZWRfaGVyZScpLCByLmdldCgnY291bnRzJykgb3IgKHIuZ2V0KCdlcnJvcicpIG9yICcnKVs6NDVdKSkKb2sgPSBzdW0oMSBmb3IgciBpbiByZXN1bHRzIGlmIHIuZ2V0KCdyZXNvbHZlZF9oZXJlJykpCnByaW50KCdcbnJlc29sdmVkIGhlcmU6ICVkIC8gJWQnICUgKG9rLCBsZW4ocmVzdWx0cykpKQpwcmludCgnd3JvdGUnLCBPVVQpCg=="""))

env = dict(os.environ, REPLAY_OUT='/kaggle/working/replay_result.json')
p = subprocess.run([sys.executable, str(RUNNER), str(TASKS), str(SNAPS), str(PD)] + sorted(PATCHES),
                   capture_output=True, text=True, env=env)
print(p.stdout)
print('STDERR', p.stderr[-1500:])
res = json.loads(Path('/kaggle/working/replay_result.json').read_text())
print('=== KAGGLE-SIDE REPLAY ===')
for r in res:
    print('%-22s resolved=%-6s counts=%s err=%s' % (r['task'], r.get('resolved_here'),
          r.get('counts'), (r.get('error') or '')[:40]))
print('resolved on Kaggle: %d / %d' % (sum(1 for r in res if r.get('resolved_here')), len(res)))
